# FPGenius Live Attention Demo

Enter a prompt and measure the same prompt-shaped Llama3-8B attention boundary on **Pure RTX 5090** and **Hybrid CPU + GPU + U50**. All values shown in Stage 2 are produced by the button press; no saved measurements are loaded.

## Stage 1 — Environment Setup

Validate the two local inputs, require an RTX 5090 and U50/XRT, load the configured xclbin, and load the local Llama3-8B model and tokenizer. This stage does not execute Attention.

In [ ]:
#********************************************************
#* Project         : FPGenius - FPT'26 Design Competition Track B
#* Filename        : demo.ipynb
#* Author          : VicChen (KUAN-HSI, CHEN)
#* Created On      : 2026/07/10
#* Function        : Load and validate the model, RTX 5090, U50, and xclbin.
#********************************************************
from __future__ import annotations

from IPython.display import Markdown, display
from demo_runtime import initialize_demo

session = initialize_demo()
setup_rows = [
    ('Model directory', session.report['model_directory']),
    ('Model format', session.report['model_format']),
    ('RTX 5090', session.report['gpu']),
    ('U50 device', session.report['u50_device']),
    ('xclbin', session.report['xclbin']),
]
setup_table = ['| Component | Detected value |', '|---|---|']
setup_table.extend(f'| {name} | {value} |' for name, value in setup_rows)
display(Markdown('### Environment ready\n\n' + '\n'.join(setup_table)))

## Stage 2 — Live Attention Benchmark

Use the left panel to enter a prompt. The local Llama3-8B weights produce first-layer post-RoPE Q/K/V, and the right panel reports the same tensors on Pure RTX 5090 and Hybrid CPU + GPU + U50. The displayed ratio uses comparable host-observed invocation envelopes on both paths. RTX CUDA-event diagnostic time is shown separately and never enters the ratio. Any device or correctness failure is displayed and stops that run.

In [ ]:
#********************************************************
#* Project         : FPGenius - FPT'26 Design Competition Track B
#* Filename        : demo.ipynb
#* Author          : VicChen (KUAN-HSI, CHEN)
#* Created On      : 2026/07/10
#* Function        : Render and execute the two-path live Attention benchmark.
#********************************************************
from __future__ import annotations

from typing import Any

import ipywidgets as widgets
from IPython.display import Markdown, display

prompt_input = widgets.Textarea(
    value='Explain why exact attention matters in one sentence.',
    description='Prompt',
    layout=widgets.Layout(width='100%', height='170px'),
)
run_button = widgets.Button(
    description='Run live comparison', button_style='primary', icon='play'
)
status_output = widgets.Output(
    layout=widgets.Layout(border='1px solid #334155', padding='12px')
)
results_output = widgets.Output(
    layout=widgets.Layout(border='1px solid #334155', padding='12px', min_height='300px')
)

def on_run(_button: Any) -> None:
    run_button.disabled = True
    status_output.clear_output(wait=True)
    results_output.clear_output(wait=True)
    try:
        with status_output:
            display(Markdown('Running both live hardware paths…'))
        result = session.benchmark(prompt_input.value)
        rows = [
            ('Host-observed invocation envelope (us)', f"{result['pure_rtx_host_invocation_us']:.3f}", f"{result['hybrid_u50_host_invocation_us']:.3f}"),
            ('Host invocation ratio (RTX / U50)', '1.000x', f"{result['host_invocation_ratio']:.3f}x"),
            ('CUDA-event diagnostic (us)', f"{result['pure_rtx_cuda_event_us']:.3f}", 'not measured'),
            ('Prompt tokens', result['prompt_tokens'], result['prompt_tokens']),
            ('Maximum absolute error', 'reference', f"{result['max_absolute_error']:.6f}"),
        ]
        lines = [
            '| Metric | Pure RTX 5090 | Hybrid CPU + GPU + U50 |',
            '|---|---:|---:|',
        ]
        lines.extend(f'| {metric} | {pure} | {hybrid} |' for metric, pure, hybrid in rows)
        with status_output:
            status_output.clear_output(wait=True)
            display(Markdown('Live run completed.'))
        with results_output:
            display(Markdown('### Live attention results\n\n' + '\n'.join(lines)))
            display(Markdown(result['measurement_note']))
    except Exception as exc:
        with status_output:
            status_output.clear_output(wait=True)
            display(Markdown(f'**Run stopped:** `{type(exc).__name__}: {exc}`'))
        raise
    finally:
        run_button.disabled = False

run_button.on_click(on_run)
left = widgets.VBox(
    [widgets.HTML('<h3>Live prompt</h3>'), prompt_input, run_button, status_output],
    layout=widgets.Layout(width='47%'),
)
right = widgets.VBox(
    [widgets.HTML('<h3>Pure RTX 5090 vs Hybrid CPU + GPU + U50</h3>'), results_output],
    layout=widgets.Layout(width='51%'),
)
display(widgets.HBox(
    [left, right],
    layout=widgets.Layout(width='100%', justify_content='space-between', align_items='flex-start'),
))